[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C32_Skills_Tools_Course/00_setup/00_environment_check.ipynb)

# 00 · 环境自检与方法论热身（MockLLM）

本课全程 **纯 Python 标准库、CPU 可跑、无需任何 API key、不依赖任何框架**。凡是需要「模型」的地方都用 **MockLLM**——一个确定性的假模型——再用 `assert` 验证你写的可扩展性子系统逻辑。

这个 notebook 做五件事：① 确认环境（连 numpy 都不需要）；② 认识本课的世界观 **能力 = 发现→选择→注入/调用 的流水线**；③ 造出本课的主角 **MockLLM**；④ 写出本课贯穿原则 **渐进披露** 的最小骨架；⑤ 立下全课纪律——**对拍 / 不变量 + assert**，并给出 **真实 Claude API 适配（无 key 自动回退）** 的统一范式。

## 1 · 环境自检

只需要标准库。**全程不联网、不需要 API key、不依赖任何框架。** 连 numpy 都不需要——本课的对象是文本解析与控制流，纯标准库足矣。

In [ ]:
import sys, platform, json, re, os
print('Python', sys.version.split()[0], '|', platform.system())
# 本课只用标准库：json/re/os/textwrap/shlex/dataclasses 等
import textwrap, shlex, dataclasses
print('标准库 json/re/os/textwrap/shlex/dataclasses 就绪')
try:
    import anthropic; print('anthropic SDK', getattr(anthropic, '__version__', '?'), '(可选, 用于接真实 Claude)')
except Exception:
    print('anthropic 未安装（可选；无它则用 MockLLM，绝不阻断）')
print('无需 API key、无需框架 —— 本课用 MockLLM。环境就绪 ✅')

## 2 · 世界观：能力 = 发现→选择→注入/调用 的流水线

一个**写死**的 agent 是「我会的就这几样」。一个**可扩展**的 agent 把能力做成一条流水线：

`磁盘上的能力 → 发现(读轻量描述) → 注册表 → 按相关性选择 → 注入/调用`。

先把这条流水线的骨架写出来——用最简单的「能力」占位（一个 name+描述+正文的字典），体会四个阶段。

In [ ]:
# 一个极简的『能力』：name + 轻量描述(常驻) + 重正文(按需)
CAPS = [
    {'name': 'git', 'desc': '处理 git 提交、分支、合并', 'body': '写提交信息时用祈使句, 50 字以内...'},
    {'name': 'sql', 'desc': '写和优化 SQL 查询', 'body': '先看表结构, 避免 SELECT *, 加合适索引...'},
    {'name': 'docs', 'desc': '写和润色文档', 'body': '先列大纲, 每段一个要点...'},
]

def discover(caps):
    '''发现：建一张轻量目录(只含 name+desc, 不含 body)。'''
    return {c['name']: c['desc'] for c in caps}

def select(task, caps, catalog):
    '''选择：按相关性(这里用关键词命中 desc)挑出相关能力的 name。'''
    hits = []
    for name, desc in catalog.items():
        # 朴素相关性：任务里出现 desc 的某个关键词
        if any(kw in task for kw in desc.replace(',', ' ').replace('、', ' ').split()):
            hits.append(name)
    return hits

def inject(names, caps):
    '''注入：只把命中能力的重正文取出来拼成上下文(渐进披露的核心)。'''
    by_name = {c['name']: c for c in caps}
    return '\n'.join(f"[{n}] {by_name[n]['body']}" for n in names)

catalog = discover(CAPS)
print('轻量目录(常驻):', catalog)
task = '帮我写一个 git 提交信息'
hits = select(task, CAPS, catalog)
ctx = inject(hits, CAPS)
print('任务:', task)
print('命中能力:', hits)
print('注入上下文:\n', ctx)
assert hits == ['git'], '只有 git 能力应被命中'
assert 'sql' not in ctx and 'SQL' not in ctx, '未命中的 sql 正文不该进上下文(渐进披露!)'
print('✅ 流水线跑通：发现→选择→注入，且未命中的重正文不占上下文')

## 3 · 造出本课的主角：MockLLM

真实里，决定「哪个 skill 相关 / 下一步调什么工具」的是大模型。本课用 **MockLLM** 代替它：一个**确定性、规则驱动**的假模型，对给定输入返回**结构正确**的输出。能力的不确定性被剥离，剩下的全是我们这套子系统的对错——最适合学协议与控制流。

下面这个 MockLLM 用「关键词→响应」规则表驱动：看到 prompt 里命中某个关键词，就返回预设的（工具调用或文本）响应。形状刻意贴近真实 tool-use。

In [ ]:
class MockLLM:
    '''确定性假模型：按规则把 prompt 映射到响应。
       规则 = [(关键词, 响应dict), ...]，第一个命中的生效；都不命中走 default。
       响应形状刻意贴近真实 tool-use：{'type':'tool_use','name','input'} 或 {'type':'text','text'}。'''
    def __init__(self, rules, default=None):
        self.rules = rules
        self.default = default or {'type': 'text', 'text': '(no rule matched)'}
        self.calls = 0                       # 记录被调次数, 便于断言
    def __call__(self, prompt):
        self.calls += 1
        text = prompt if isinstance(prompt, str) else json.dumps(prompt, ensure_ascii=False)
        for kw, resp in self.rules:
            if kw in text:
                return json.loads(json.dumps(resp))   # 深拷贝, 避免被调用方改坏规则
        return json.loads(json.dumps(self.default))

llm = MockLLM(rules=[
    ('提交', {'type': 'tool_use', 'name': 'git_commit', 'input': {'message': 'feat: x'}}),
    ('你好', {'type': 'text', 'text': '你好，我能帮你处理 git / sql / 文档。'}),
])
r1 = llm('帮我写一个 git 提交信息')
r2 = llm('你好呀')
r3 = llm('随便说点啥')
print('含『提交』 ->', r1)
print('打招呼   ->', r2)
print('未命中   ->', r3)
assert r1['type'] == 'tool_use' and r1['name'] == 'git_commit'
assert r2['type'] == 'text'
assert r3 == {'type': 'text', 'text': '(no rule matched)'}
assert llm.calls == 3
print('✅ MockLLM 工作正常：确定性、可断言、形状贴近真实 tool-use')

## 4 · 本课贯穿原则：渐进披露(progressive disclosure)

这是整门课最重要的设计原则：**平时只把能力的轻量描述放进上下文，真正用到时才加载完整内容**。

为什么？上下文窗口有限。若把 50 个 skill 的完整正文全塞进去，既贵又挤掉了真正的任务内容。渐进披露让你能『拥有』很多能力，却只为『用到的』那几个付上下文成本。

下面量化体会一下：对比『全量加载』与『按需加载』的上下文字符数。

In [ ]:
# 假设有 5 个 skill, 每个描述很短、正文较长
SKILLS = [{'name': f's{i}', 'desc': f'技能{i}的一句话描述',
           'body': f'技能{i}的详细正文' + 'x' * 200} for i in range(5)]

def full_load(skills):
    '''反例：把所有 skill 的正文全部加载(臃肿)。'''
    return '\n'.join(s['body'] for s in skills)

def lazy_load(skills, relevant_names):
    '''渐进披露：描述常驻(轻), 只对命中的 skill 加载正文。'''
    catalog = '\n'.join(f"{s['name']}: {s['desc']}" for s in skills)  # 轻量, 常驻
    by_name = {s['name']: s for s in skills}
    bodies = '\n'.join(by_name[n]['body'] for n in relevant_names)      # 重, 按需
    return catalog + '\n' + bodies

full = full_load(SKILLS)
lazy = lazy_load(SKILLS, relevant_names=['s2'])   # 只有 s2 相关
print(f'全量加载上下文字符数: {len(full)}')
print(f'按需加载上下文字符数: {len(lazy)}  (描述常驻 + 只加载 s2 正文)')
assert len(lazy) < len(full), '渐进披露应当显著更省上下文'
# 描述都在(发现得到), 但只有 s2 的正文进了上下文
assert 's2: ' in lazy and '技能2的详细正文' in lazy
assert '技能0的详细正文' not in lazy, '未命中的 s0 正文不该被加载'
print('✅ 渐进披露：描述常驻保证可发现, 正文按需保证省上下文 —— 全课贯穿此原则')

## 5 · 立纪律：不变量 + assert，以及真实 Claude API 适配（无 key 自动回退）

本课的裁判是**不变量**：你写的每个子系统在合法输入上做对、在每一类异常输入上**正确处理而非崩溃**。逻辑正确则 assert 通过；assert 通过则可把零件原样搬到真实 Claude Code / MCP。

本课特例：每个 notebook 都给出**接真实 Claude 的适配代码**，并约定一个统一范式——**有 `ANTHROPIC_API_KEY` 走真实 `messages.create`，没有就自动回退 MockLLM**，让代码永远跑得通、绝不阻断学习。下面把这个范式写成一个全课复用的工厂。

In [ ]:
def make_llm(rules=None, default=None, model='claude-sonnet-4-6'):
    '''统一 LLM 工厂：有 ANTHROPIC_API_KEY 且装了 anthropic -> 真实 Claude；否则 -> MockLLM。
       返回一个 callable(prompt_or_messages) -> {'type':'text'/'tool_use', ...}。
       本课所有 notebook 共用这个范式：你写的 scaffold 不必改, 只换模型这一处。'''
    key = os.environ.get('ANTHROPIC_API_KEY')
    if key:
        try:
            import anthropic
            client = anthropic.Anthropic()           # 读 ANTHROPIC_API_KEY
            def real_llm(prompt):
                msgs = prompt if isinstance(prompt, list) else [{'role': 'user', 'content': prompt}]
                resp = client.messages.create(model=model, max_tokens=1024, messages=msgs)
                # 把 Anthropic 响应规约成本课统一形状
                for block in resp.content:
                    if block.type == 'tool_use':
                        return {'type': 'tool_use', 'name': block.name, 'input': block.input}
                txt = ''.join(b.text for b in resp.content if b.type == 'text')
                return {'type': 'text', 'text': txt}
            print(f'[make_llm] 使用真实 Claude: {model}')
            return real_llm
        except Exception as e:
            print(f'[make_llm] 真实 Claude 不可用({type(e).__name__}), 回退 MockLLM')
    # 无 key 或异常 -> MockLLM, 绝不阻断
    print('[make_llm] 无 API key, 使用 MockLLM (确定性, 可断言)')
    return MockLLM(rules or [], default=default)

# 本课环境无 key, 所以这里拿到的是 MockLLM —— 但形状与真实一致, 学完换模型即可
llm = make_llm(rules=[('提交', {'type': 'tool_use', 'name': 'git_commit', 'input': {'message': 'feat: x'}})])
out = llm('帮我写 git 提交信息')
print('输出:', out)
assert out['type'] == 'tool_use' and out['name'] == 'git_commit'
print('✅ make_llm 就位：无 key 回退 MockLLM, 有 key 走真实 Claude —— 全课共用此范式')

✅ 检查全部通过即环境就绪、方法论到位。

**本课的契约**：你写的每个零件（skill 加载器、命令路由、MCP server/client、插件加载器、完整系统）都用**不变量 + assert** 验证；逻辑正确则 assert 通过，assert 通过则可把它接到真实 Claude Code / MCP，并把 `MockLLM(...)` 换成 `make_llm(...)` 直接迁移。

**接下来五个模块**：01 skill 加载 → 02 slash 命令 → 03 MCP server → 04 工具打包 → 05 完整系统。每一步都建立在「发现→选择→注入/调用」这条流水线与「渐进披露」这个原则上。

下一站：**模块 01 · Skill 定义与加载**。